# 05 — L1 Judge-Model Swap v2 (Gemini 2.5 Pro on anchor cell)

Second alternative judge for the L1-axis. Mirror of `04_L1_swap_claude.ipynb` (which used Claude Sonnet 4.5) but with Gemini 2.5 Pro — Arena-Hard v2.0's other officially supported judge alongside GPT-4.1.

**Why this matters**: with only one alternative judge (Claude), the L1-axis finding can be dismissed as "Claude-specific". With a second alternative judge — and especially one that Arena-Hard's own pipeline officially supports — the L1-axis becomes a *structural* axis claim, not a single-vendor observation.

**Config (identical to `04_L1_swap_claude.ipynb` except judge):**
- Judge model: `gemini-2.5-pro` (Arena-Hard v2.0 official secondary judge)
- Temperature: 0.0
- Max output tokens: 4096
- Anchor cell only: `C1F1D1` (Arena-Hard's default rubric + reasoning + no scope)
- 500 prompts × 2 rounds (A-then-B, B-then-A) = 1,000 calls
- Resumable: skip uids already in output JSONL

**Cost estimate**: ~£10–25 at 2025 Gemini pricing. Runtime ~30–60 min with `max_workers=4`.

**Outputs**: `data/L1_swap_v2/C1F1D1.jsonl` — same schema as `data/L1_swap/C1F1D1.jsonl`.

**After this notebook runs**: update `10_analysis.ipynb` by adding `'data/L1_swap_v2/C1F1D1.jsonl'` to the data-source list near the top, then re-execute the analysis. This produces an extended `master_ladder.csv` with three judges (gpt-4.1, claude-sonnet-4-5, gemini-2.5-pro) and lets the next notebook (`11_inter_judge_agreement.ipynb`) compute pairwise κ across all three pairs.

**Setup**: install the Gemini SDK and add the API key to `.env`:
```bash
pip install google-genai
echo 'GEMINI_API_KEY=<your-key>' >> ./.env
```

In [1]:
import concurrent.futures
import json
import os
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from tqdm.auto import tqdm

from _prompts import get_system_prompt, get_user_prompt

## Config

In [2]:
JUDGE_MODEL = "gemini-2.5-pro"
TEMPERATURE = 0.0
MAX_TOKENS = 16000        # full output budget (thinking + visible). Was 4096 — too low for a reasoning model.
THINKING_BUDGET = 8192    # NEW: cap private thinking at 8k tokens so 8k+ always remain for the visible verdict.
ANCHOR_CELL = "C1F1D1"
MAX_WORKERS = 12          # NEW: bumped from 4 → 12 for ~3x throughput (paid-tier Gemini handles this).
MAX_RETRIES = 8

REPO_ROOT = Path(".")
DATA_DIR = Path("data/L1_swap_v2")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = DATA_DIR / f"{ANCHOR_CELL}.jsonl"

# Same regex as _pipeline.py (Arena-Hard v2.0 default)
PRIMARY_REGEX = re.compile(r"\[\[([AB<>=]+)\]\]")
FALLBACK_REGEX = re.compile(r"\[([AB<>=]+)\]")

load_dotenv(REPO_ROOT / ".env")
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise RuntimeError("GEMINI_API_KEY (or GOOGLE_API_KEY) not found in .env")

client = genai.Client(api_key=api_key)
print(f"Judge: {JUDGE_MODEL} | T={TEMPERATURE} | max_tokens={MAX_TOKENS} | thinking_budget={THINKING_BUDGET} | workers={MAX_WORKERS}")
print(f"Output: {OUTPUT_PATH}")

Judge: gemini-2.5-pro | T=0.0 | max_tokens=16000 | thinking_budget=8192 | workers=12
Output: data/L1_swap_v2/C1F1D1.jsonl


## Load preprocessed pairs

In [3]:
pairs = pd.read_json("data/preprocessed_pairs.jsonl", lines=True)
print(f"Loaded {len(pairs)} pairs")
print(pairs[['uid', 'category', 'subcategory', 'language']].head())

Loaded 500 pairs
                uid     category subcategory language
0  2edbb5f36f5b42be  hard_prompt      coding  English
1  ec71c09662a64365  hard_prompt      coding  English
2  d5cdf24c4e614beb  hard_prompt      coding  English
3  dfc9be7c176d46bb  hard_prompt      coding  English
4  666d2acdd7d64e17  hard_prompt      coding  English


## Judge call wrapper

In [4]:
import random  # for jittered backoff

def parse_verdict(text: str):
    """Extract verdict tag - primary then fallback regex."""
    upper = text.upper()
    matches = [m for m in PRIMARY_REGEX.findall(upper) if m]
    if matches:
        return matches[-1].strip("\n")
    matches = [m for m in FALLBACK_REGEX.findall(upper) if m]
    if matches:
        return matches[-1].strip("\n")
    return None


def call_gemini_judge(system_prompt, user_prompt):
    """Single Gemini judge call with thinking_budget cap + jittered retry on transient errors."""
    delay = 2.0
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.models.generate_content(
                model=JUDGE_MODEL,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    temperature=TEMPERATURE,
                    max_output_tokens=MAX_TOKENS,
                    thinking_config=types.ThinkingConfig(
                        thinking_budget=THINKING_BUDGET,
                    ),
                ),
            )
            break
        except Exception as e:
            err = str(e).lower()
            is_transient = any(s in err for s in ("429", "rate", "quota", "503", "overload", "unavailable", "resource_exhausted", "500"))
            if is_transient and attempt < MAX_RETRIES - 1:
                # jittered exponential backoff (caps at 60s)
                time.sleep(delay + random.random())
                delay = min(delay * 1.7, 60.0)
                continue
            raise
    raw = resp.text or ""
    verdict = parse_verdict(raw)
    return {
        "raw": raw,
        "verdict": verdict,
        "parse_success": verdict is not None,
        "input_tokens": getattr(resp.usage_metadata, "prompt_token_count", None),
        "output_tokens": getattr(resp.usage_metadata, "candidates_token_count", None),
        "thinking_tokens": getattr(resp.usage_metadata, "thoughts_token_count", None),
        "cached_tokens": getattr(resp.usage_metadata, "cached_content_token_count", None),
        "total_tokens": getattr(resp.usage_metadata, "total_token_count", None),
    }


def run_judge_pair(cell_id, question, candidate_response, baseline_response):
    """Two-round bidirectional swap (Arena-Hard default)."""
    system_prompt = get_system_prompt(cell_id)
    # Round 1: A=baseline, B=candidate
    user_r1 = get_user_prompt(cell_id, question, baseline_response, candidate_response)
    r1 = call_gemini_judge(system_prompt, user_r1)
    # Round 2: positions swapped
    user_r2 = get_user_prompt(cell_id, question, candidate_response, baseline_response)
    r2 = call_gemini_judge(system_prompt, user_r2)
    return {"round1": r1, "round2": r2}

## Run — two-round × anchor cell × 500 prompts (resumable)

In [ ]:
# Resume: rewrite output to keep only fully-valid rows, then skip those UIDs.
# This handles the case where a previous run wrote rows with one or both rounds
# failing parse — those rows are dropped here and the UIDs become retry candidates.
done_uids = set()
if OUTPUT_PATH.exists():
    valid_rows = []
    n_seen = n_dropped = 0
    with open(OUTPUT_PATH) as f:
        for line in f:
            rec = json.loads(line)
            n_seen += 1
            r1_ok = rec.get("round1", {}).get("parse_success", False)
            r2_ok = rec.get("round2", {}).get("parse_success", False)
            if r1_ok and r2_ok:
                valid_rows.append(rec)
            else:
                n_dropped += 1
    # rewrite file atomically (write to .tmp, then os.replace) so an interrupted
    # rewrite cannot lose previously-valid rows. [L1_GEMINI_PATCH round 3]
    import os
    by_uid = {r["uid"]: r for r in valid_rows}
    tmp_path = OUTPUT_PATH.with_suffix(".jsonl.tmp")
    with open(tmp_path, "w", encoding="utf-8") as f:
        for rec in by_uid.values():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    os.replace(tmp_path, OUTPUT_PATH)
    done_uids = set(by_uid.keys())
    print(f"Resuming - {len(done_uids)} uids fully valid (was {n_seen} rows; dropped {n_dropped} with at least one invalid round)")

to_run = pairs[~pairs["uid"].isin(done_uids)]
rows = to_run.to_dict("records")
print(f"To run: {len(rows)} prompts x 2 rounds = {len(rows) * 2} calls")

def process_one(row):
    try:
        result = run_judge_pair(
            ANCHOR_CELL,
            question=row["prompt"],
            candidate_response=row["candidate_response"],
            baseline_response=row["baseline_response"],
        )
        return {
            "uid": row["uid"],
            "spec_cell_id": ANCHOR_CELL,
            "round1": result["round1"],
            "round2": result["round2"],
            "model": JUDGE_MODEL,
            "timestamp": time.time(),
        }
    except Exception as e:
        print(f"error uid={row['uid']}: {e}")
        return None

start = time.time()
print(f"\nStart: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
with open(OUTPUT_PATH, "a", encoding="utf-8") as f:
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_one, row) for row in rows]
        for fut in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc=f"{JUDGE_MODEL} x{MAX_WORKERS}"):
            rec = fut.result()
            if rec is not None:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                f.flush()

elapsed = timedelta(seconds=int(time.time() - start))
print(f"\nDone. Elapsed: {elapsed}")

Resuming - 129 uids fully valid (was 129 rows; dropped 0 with at least one invalid round)
To run: 371 prompts x 2 rounds = 742 calls

Start: 2026-05-25 20:28:26


gemini-2.5-pro x12:   0%|          | 0/371 [00:00<?, ?it/s]

error uid=206b01d99e3c4016: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_requests_per_model_per_day, limit: 1000, model: gemini-2.5-pro\nPlease retry in 3h29m36.219082357s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_requests_per_model_per_day', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel', 'quotaDimensions': {'model': 'gemini-2.5-pro', 'location': '

## Verify outputs

In [ ]:
out = pd.read_json(OUTPUT_PATH, lines=True)
print(f"Total rows: {len(out)}")

r1_invalid = out["round1"].apply(lambda r: not r.get("parse_success", False)).sum()
r2_invalid = out["round2"].apply(lambda r: not r.get("parse_success", False)).sum()
print(f"Invalid round1: {r1_invalid} | Invalid round2: {r2_invalid}")
print(f"Total invalid: {r1_invalid + r2_invalid} / {len(out) * 2} ({(r1_invalid + r2_invalid) / (len(out) * 2) * 100:.2f}%)")

# Verdict distribution
from collections import Counter
verdicts_r1 = [r.get("verdict") for r in out["round1"]]
verdicts_r2 = [r.get("verdict") for r in out["round2"]]
print(f"\nRound 1 verdict distribution: {Counter(verdicts_r1)}")
print(f"Round 2 verdict distribution: {Counter(verdicts_r2)}")

## Next steps

1. Update `10_analysis.ipynb` to include the new judge data:
   - Find the cell that loads L1 swap data.
   - Add `'data/L1_swap_v2/C1F1D1.jsonl'` to the list (with a new variant label like `L1_gemini`).
   - Re-execute the full notebook — bucket assignment, master_ladder, axis fingerprints, language amplifier all extend automatically.
2. Create `11_inter_judge_agreement.ipynb` (next) to compute pairwise Cohen's κ across all three judge pairs (GPT-4.1, Claude, Gemini).